# Qwen3-Embedding-0.6B + CatBoost: three controlled experiments

This notebook runs end-to-end without manual cells:

1. lexical name features;
2. names plus cached Qwen item embeddings;
3. names, embeddings, and structured JSON attributes.

All three models use the same component-disjoint validation split.
Final models, predictions, reports, logs, the selected attribute
keys, and the float16 embedding cache are saved in
`/kaggle/working/embedding_boosting`.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
PROJECT_ROOT = WORKING_ROOT / 'product_matching'
OUTPUT_DIR = WORKING_ROOT / 'embedding_boosting'
EXPECTED_BUNDLE_SHA256 = '0322601bdfa9a04173984b0d363d474002743f6f38f14483ae6159100d9df7de'

def exactly_one(filename):
    candidates = list(INPUT_ROOT.glob(f'**/{filename}'))
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {filename}, found {candidates}')
    return candidates[0]

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

items_path = exactly_one('items_human.parquet')
matches_path = exactly_one('matches.parquet')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
bundle_files = list(INPUT_ROOT.glob('**/product_matching_training_code.zip'))
bundle_directories = [
    path for path in INPUT_ROOT.glob('**/product_matching_training_code')
    if path.is_dir()
]
bundle_candidates = bundle_files + bundle_directories
if len(bundle_candidates) != 1:
    raise RuntimeError(f'Expected one code bundle, found {bundle_candidates}')
bundle_path = bundle_candidates[0]
if bundle_path.is_file():
    if sha256(bundle_path) != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Attached source bundle hash does not match this notebook')
    with zipfile.ZipFile(bundle_path) as archive:
        for member in archive.namelist():
            path = PurePosixPath(member)
            if path.is_absolute() or '..' in path.parts:
                raise RuntimeError(f'Unsafe bundle member: {member}')
        archive.extractall(PROJECT_ROOT)
else:
    shutil.copytree(bundle_path, PROJECT_ROOT, dirs_exist_ok=True)
print('items:', items_path)
print('matches:', matches_path)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## Install the pinned experiment environment

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed',
     '-r', str(PROJECT_ROOT / 'requirements-embedding-boosting.txt')],
    check=True,
)

## Run all experiments and persist every artifact

In [ ]:
command = [
    sys.executable, '-u', str(PROJECT_ROOT / 'src/embedding_boosting.py'),
    '--items', str(items_path),
    '--matches', str(matches_path),
    '--config', str(PROJECT_ROOT / 'configs/embedding_boosting.json'),
    '--output-dir', str(OUTPUT_DIR),
]
print('$', ' '.join(command), flush=True)
with (WORKING_ROOT / 'embedding_boosting_console.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(f'Experiment process failed with exit code {return_code}')
print((OUTPUT_DIR / 'experiment_comparison.csv').read_text(encoding='utf-8'))
print('COMPLETED:', OUTPUT_DIR)